## Readme
- 该版本在基础模型上增加了与TOES的对比
- 如果只是载入模型而不需要训练的情况下，直接执行Cell01-07+Cell10-11及其后面Cell12-14的可视化（可选）；即跳过Cell08-09.

## Cell 01 — 配置与设备/随机种子

In [2]:
# Cell 01: Config + device + seed (standalone)

import os, yaml, random
from types import SimpleNamespace
from pathlib import Path
import numpy as np
import torch

def to_ns(d):
    if isinstance(d, dict): return SimpleNamespace(**{k: to_ns(v) for k,v in d.items()})
    if isinstance(d, list): return [to_ns(x) for x in d]
    return d

def load_config(path="config/config_ergodic.yaml"):
    cfg = None
    p = Path(path)
    if p.exists():
        with open(p, "r") as f:
            cfg = yaml.safe_load(f)
    else:
        # Fallback 默认配置（独立运行）
        cfg = {
            "data": {
                "data_dir": "data/ergodic_dataset",   # 相对当前notebook目录
                "trajectory_len": 101,
                "robot_state_dim": 4,
                "distribution_dim": [32,32],
                "validation_split": 0.1,
                "num_workers": 0,
                "shuffle_dataset": True,
                "seed": 42
            },
            "training": {
                "batch_size": 64,
                "device": "cuda",
                "seed": 42
            },
            "diffusion": {
                "beta_min": 0.1,
                "beta_max": 20.0,
                "steps": 100
            },
            "normalizer": {
                "robot_state": {"mean": [0.0,0.0,0.0,0.0], "std": [1.0,1.0,1.0,1.0]}
            }
        }
    cfg = to_ns(cfg)

    # 兼容顶层常用字段
    cfg.data_dir = cfg.data.data_dir
    cfg.trajectory_len = cfg.data.trajectory_len
    cfg.robot_state_dim = cfg.data.robot_state_dim
    cfg.distribution_dim = cfg.data.distribution_dim

    # 规范化 mean/std 为 tensor
    mean = torch.as_tensor(cfg.normalizer.robot_state.mean, dtype=torch.float32)
    std  = torch.as_tensor(cfg.normalizer.robot_state.std,  dtype=torch.float32)
    std  = torch.where(std==0, torch.tensor(1.0), std)
    cfg.normalizer.robot_state.mean = mean
    cfg.normalizer.robot_state.std  = std
    return cfg

config = load_config()  # 如需自定义，改这里路径
device = torch.device(config.training.device if torch.cuda.is_available() else "cpu")

# 随机种子
seed = int(getattr(config.training, "seed", 42))
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

print("Device:", device)
print("Data dir:", config.data_dir)


Device: cuda
Data dir: /home/songxy/code/time-optimal-ergodic-search/experiments/bias_search/ergodic_dataset


## Cell 02 — 数据集与 DataLoader（独立实现）
获取训练/验证 DataLoader（Dataset 已含 padding）

In [ ]:
# Data module (final): robust tensorization with torch.as_tensor, no from_numpy

import os, json, numpy as np, torch
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler

def as_tensor_f32(x):
    return x.to(torch.float32) if isinstance(x, torch.Tensor) else torch.as_tensor(x, dtype=torch.float32)

class ErgodicDataset(Dataset):
    def __init__(self, data_dir, transform=None, max_trajectory_len=101, use_index=True):
        self.data_dir = data_dir
        self.transform = transform
        self.max_trajectory_len = max_trajectory_len
        self.distributions_dir = os.path.join(data_dir, 'distributions')
        self.trajectories_dir  = os.path.join(data_dir, 'trajectories')
        self.data_pairs = []

        index_path = os.path.join(data_dir, 'dataset_index.json')
        if use_index and os.path.exists(index_path):
            try:
                with open(index_path, 'r') as f:
                    idx = json.load(f)
                if isinstance(idx, dict) and 'distributions' in idx and 'trajectories' in idx:
                    dist_id_to_file = {d: f"{d}.json" for d in idx['distributions']}
                    for t in idx['trajectories']:
                        parts = t.split('_')
                        if len(parts) >= 2:
                            dist_id = f"{parts[0]}_{parts[1]}"
                            if dist_id in dist_id_to_file:
                                self.data_pairs.append({
                                    'distribution_file': dist_id_to_file[dist_id],
                                    'trajectory_file': f"{t}.json"
                                })
                else:
                    self._build_index(index_path)
            except Exception:
                self._build_index(index_path)
        else:
            self._build_index(index_path)

    def _build_index(self, index_path):
        dist_files = [f for f in os.listdir(self.distributions_dir) if f.endswith('.json')]
        dist_id_to_file = {}
        for df in dist_files:
            try:
                with open(os.path.join(self.distributions_dir, df), 'r') as f:
                    d = json.load(f)
                dist_id_to_file[d['id']] = df
            except Exception:
                pass
        for tf in os.listdir(self.trajectories_dir):
            if tf.endswith('.json'):
                try:
                    with open(os.path.join(self.trajectories_dir, tf), 'r') as f:
                        t = json.load(f)
                    did = t['distribution_id']
                    if did in dist_id_to_file:
                        self.data_pairs.append({
                            'distribution_file': dist_id_to_file[did],
                            'trajectory_file': tf
                        })
                except Exception:
                    pass
        try:
            with open(index_path, 'w') as f:
                json.dump(self.data_pairs, f)
        except Exception:
            pass

    def __len__(self):
        return len(self.data_pairs)

    def _generate_distribution_grid(self, dist_data, grid_size=(32,32)):
        bounds = dist_data.get('bounds', [[0,3],[0,3]])
        centers = np.asarray(dist_data['params']['centers'])
        covs    = np.asarray(dist_data['params']['covs'])
        weights = np.asarray(dist_data['params']['weights'])
        n = int(dist_data['params']['n_gaussians'])
        x = np.linspace(bounds[0][0], bounds[0][1], grid_size[0])
        y = np.linspace(bounds[1][0], bounds[1][1], grid_size[1])
        X, Y = np.meshgrid(x, y)
        Z = np.zeros_like(X, dtype=np.float64)
        for i in range(n):
            cx, cy = centers[i]
            c = covs[i]
            if np.isscalar(c):
                sx, sy = c, c
            elif len(np.shape(c)) == 1:
                sx, sy = c[0], c[1]
            else:
                sx, sy = np.sqrt(np.diag(c))
            Z += weights[i] * np.exp(-(((X-cx)**2)/(2*sx**2 + 1e-8) + ((Y-cy)**2)/(2*sy**2 + 1e-8)))
        Z /= (Z.max() + 1e-8)
        return Z

    def _process_trajectory(self, traj_data):
        states = np.asarray(traj_data['states'], dtype=np.float64)
        pos = states[:, :2]
        heading = states[:, 2:3] if states.shape[1] > 2 else np.zeros((states.shape[0],1))
        if states.shape[1] <= 3:
            dt = float(traj_data['time_step'])
            vel = np.zeros((states.shape[0],1))
            if states.shape[0] > 1:
                d = np.linalg.norm(np.diff(pos, axis=0), axis=1)
                vel = np.vstack([np.zeros((1,1)), (d/dt)[:,None]])
        else:
            vel = states[:, 3:4]
        rs = np.hstack([pos, heading, vel])  # [T,4]
        controls = np.asarray(traj_data.get('controls', np.zeros((rs.shape[0],2))), dtype=np.float64)

        T = self.max_trajectory_len
        if len(rs) > T:
            idx = np.linspace(0, len(rs)-1, T).astype(int)
            rs = rs[idx]; controls = controls[idx] if len(controls)>0 else controls
        elif len(rs) < T:
            pad = T - len(rs)
            rs = np.vstack([rs, np.zeros((pad, rs.shape[1]))])
            if len(controls) > 0:
                controls = np.vstack([controls, np.zeros((pad, controls.shape[1]))])
        return rs, controls, float(traj_data['time_step']), float(traj_data['total_time']), float(traj_data['ergodic_metric']), float(traj_data['gamma'])

    def __getitem__(self, idx):
        pair = self.data_pairs[idx]
        with open(os.path.join(self.distributions_dir, pair['distribution_file']), 'r') as f:
            dist_data = json.load(f)
        with open(os.path.join(self.trajectories_dir, pair['trajectory_file']), 'r') as f:
            traj_data = json.load(f)

        distribution_grid = self._generate_distribution_grid(dist_data)   # np [H,W]
        rs, controls, time_step, total_time, ergodic_metric, gamma = self._process_trajectory(traj_data)

        n_gauss = int(dist_data['params']['n_gaussians'])
        centers = np.zeros((10,2)); covs = np.zeros(10); weights = np.zeros(10)
        c = np.asarray(dist_data['params']['centers']); v = np.asarray(dist_data['params']['covs']); w = np.asarray(dist_data['params']['weights'])
        centers[:n_gauss] = c[:n_gauss]; covs[:n_gauss] = v[:n_gauss]; weights[:n_gauss] = w[:n_gauss]

        sample = {
            "distribution_id": dist_data['id'],
            "trajectory_id": os.path.splitext(pair['trajectory_file'])[0],
            "distribution": distribution_grid,
            "robot_state": rs[0],
            "trajectories": rs,
            "controls": controls,
            "time_step": time_step,
            "total_time": total_time,
            "ergodic_metric": ergodic_metric,
            "gamma": gamma,
            "gaussian_params": {"n_gaussians": n_gauss, "centers": centers, "covs": covs, "weights": weights}
        }

        if self.transform is not None:
            sample = self.transform(sample)  # 允许返回 numpy 或 torch.Tensor

        # 统一张量化（仅使用 as_tensor_f32）
        sample["distribution"] = as_tensor_f32(sample["distribution"])
        if sample["distribution"].ndim == 2:
            sample["distribution"] = sample["distribution"].unsqueeze(0)  # [1,H,W]
        sample["robot_state"]  = as_tensor_f32(sample["robot_state"])
        sample["trajectories"] = as_tensor_f32(sample["trajectories"])
        sample["controls"]     = as_tensor_f32(sample["controls"])

        gp = sample.get("gaussian_params", {})
        if isinstance(gp, dict):
            for k in ("centers","covs","weights"):
                if k in gp: gp[k] = as_tensor_f32(gp[k])
            sample["gaussian_params"] = gp

        return sample

def build_loaders(data_dir, batch_size=64, max_trajectory_len=101, val_split=0.1, num_workers=0, seed=42, transform=None):
    # 首次建议 num_workers=0，确认正常后可调大
    ds = ErgodicDataset(data_dir=data_dir, transform=transform, max_trajectory_len=max_trajectory_len)
    N = len(ds); idx = list(range(N))
    split = int(np.floor(val_split * N))
    rng = np.random.RandomState(seed); rng.shuffle(idx)
    val_idx, train_idx = idx[:split], idx[split:]
    train_loader = DataLoader(ds, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx),
                              num_workers=num_workers, pin_memory=True, drop_last=False)
    val_loader   = DataLoader(ds, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx),
                              num_workers=num_workers, pin_memory=True, drop_last=False)
    return train_loader, val_loader

print("ErgodicDataset ready (no from_numpy; unified torch.as_tensor).")


## Cell 03 —  SDE 与训练/评估通用工具

In [ ]:
# Cell 03: VPSDE_linear + training utilities

import math
import torch
import torch.nn as nn

class VPSDE_linear:
    def __init__(self, beta_min=0.1, beta_max=20.0):
        self._beta_min = float(beta_min)
        self._beta_max = float(beta_max)
    @property
    def T(self): return 1.0
    def marginal_prob(self, x, t):
        # x_t = exp(mean_log_coeff)*x0 + std*z
        shape = x.shape
        t = t.view(-1, *([1]*(len(shape)-1)))
        mlc = -0.25 * t**2 * (self._beta_max - self._beta_min) - 0.5*self._beta_min*t
        mean = torch.exp(mlc) * x
        std  = torch.sqrt(torch.clamp(1 - torch.exp(2*mlc), min=1e-6))
        return mean, std

def time_weighted_masked_mse(pred, target, gamma=4.0, eps=1e-8):
    # pred/target: [B,T,D]；仅有效时刻参与；末段权重大
    B,T,D = target.shape
    mask = (target.abs().sum(dim=-1) > eps).float()            # [B,T]
    t = torch.arange(T, device=target.device).float().view(1,T)
    w = (t / max(T-1,1))**gamma
    w = (w*mask); w = w/(w.sum(dim=1, keepdim=True)+eps)
    diff2 = ((pred-target)**2).sum(dim=-1)
    return (((diff2*w).sum(dim=1)).mean())/D

def last_valid_idx_xy(xy, eps=1e-8):
    # xy: [B,T,2] or [T,2]
    if xy.ndim == 2:
        valid = ~(xy.abs().sum(dim=-1) <= eps)
        k = torch.nonzero(valid)[-1].item() if valid.any() else xy.shape[0]-1
        return k
    valid = ~(xy.abs().sum(dim=-1) <= eps)                     # [B,T]
    idx = []
    for b in range(xy.shape[0]):
        v = valid[b]; k = torch.nonzero(v)[-1].item() if v.any() else xy.shape[1]-1
        idx.append(k)
    return torch.tensor(idx, device=xy.device, dtype=torch.long)


## Cell 04 — Encoder

In [ ]:
# Cell 04: ErgodicEncoder

import torch.nn.functional as F
import torch.nn as nn

class ErgodicEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.hidden_dim = int(getattr(cfg, 'hidden_dim', getattr(getattr(cfg,'model',object()),'hidden_dim',192)))
        state_dim = int(cfg.robot_state_dim)
        H,W = cfg.distribution_dim

        self.robot_encoder = nn.Sequential(
            nn.Linear(state_dim, self.hidden_dim//2), nn.GELU(),
            nn.Linear(self.hidden_dim//2, self.hidden_dim)
        )
        self.distribution_encoder = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.GELU(),
            nn.Conv2d(16,32,3,padding=1,stride=2), nn.GELU(),
            nn.Conv2d(32,64,3,padding=1,stride=2), nn.GELU()
        )
        dist_h, dist_w = H//4, W//4
        self.distribution_projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(dist_h*dist_w*64, self.hidden_dim), nn.GELU(),
            nn.Linear(self.hidden_dim, self.hidden_dim)
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(self.hidden_dim),
            nn.Linear(self.hidden_dim, self.hidden_dim), nn.GELU(),
            nn.LayerNorm(self.hidden_dim)
        )

    def forward(self, inputs):
        rs = inputs['robot_state']            # 标准化
        dist = inputs['distribution']
        rs_enc   = self.robot_encoder(rs)
        dist_feat= self.distribution_encoder(dist)
        dist_enc = self.distribution_projector(dist_feat)
        enc = self.fusion(rs_enc + dist_enc)
        return {"encoding": enc, "robot_state": rs}


## Cell 05 — DiT/Decoder（仅采样器保留允许的导入）

In [ ]:
# Cell 05: DiT blocks + Decoder (sampling via dpm_solver)

import torch
import torch.nn as nn
from timm.models.layers import Mlp
from models.diffusion_utils.sampling import dpm_sampler  # 允许的唯一外部导入

class TimestepEmbedder(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.half = hidden_dim//2
        self.emb = nn.Linear(hidden_dim, hidden_dim)
    def forward(self, t):
        freqs = torch.exp(-math.log(10000)*torch.arange(0,self.half,device=t.device)/self.half)
        x = torch.cat([torch.cos(t[:,None]*freqs[None,:]),
                       torch.sin(t[:,None]*freqs[None,:])], dim=-1)
        return self.emb(x)

class DiTBlock(nn.Module):
    def __init__(self, hidden_dim, heads, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.sa = nn.MultiheadAttention(hidden_dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.ca = nn.MultiheadAttention(hidden_dim, heads, dropout=dropout, batch_first=True)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.mlp = Mlp(in_features=hidden_dim, hidden_features=int(hidden_dim*4), act_layer=nn.GELU, drop=dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, context=None):
        h = self.norm1(x); h = self.sa(h.unsqueeze(1), h.unsqueeze(1), h.unsqueeze(1))[0].squeeze(1)
        x = x + self.drop(h)
        h = self.norm2(x); h = self.ca(h.unsqueeze(1), context.unsqueeze(1), context.unsqueeze(1))[0].squeeze(1)
        x = x + self.drop(h)
        h = self.norm3(x); h = self.mlp(h); x = x + self.drop(h)
        return x

class FinalLayer(nn.Module):
    def __init__(self, hidden_dim, output_dim):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim)
        self.proj = nn.Linear(hidden_dim, output_dim)
    def forward(self, x, t_emb):
        return self.proj(self.norm(x))

class ErgodicDiT(nn.Module):
    def __init__(self, output_dim, hidden_dim, depth, heads, dropout=0.1, model_type="x_start"):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.model_type = model_type     # 必须：供 dpm_sampler 读取
        self.traj_proj = nn.Sequential(
            nn.Linear(output_dim, hidden_dim * 2), nn.GELU(), nn.Linear(hidden_dim * 2, hidden_dim)
        )
        self.t_embedder = TimestepEmbedder(hidden_dim)
        self.blocks = nn.ModuleList([DiTBlock(hidden_dim, heads, dropout=dropout) for _ in range(depth)])
        self.final = FinalLayer(hidden_dim, output_dim)

    def forward(self, x, t, context, conditions=None):
        # 可选：采样时强制第 0 步（训练时不要用）
        if isinstance(conditions, dict) and 0 in conditions and conditions[0] is not None:
            B = x.shape[0]
            # 仅在采样时传入 conditions；训练路径不要传
            pass
        h = self.traj_proj(x)
        t_emb = self.t_embedder(t)
        h = h + t_emb
        for blk in self.blocks:
            h = blk(h, context)
        return self.final(h, t_emb)

class ErgodicDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.T = int(getattr(config, 'trajectory_len', getattr(getattr(config,'data',object()), 'trajectory_len', 101)))
        self.D = int(getattr(config, 'robot_state_dim', getattr(getattr(config,'data',object()), 'robot_state_dim', 4)))
        hd = int(getattr(config, 'hidden_dim', getattr(getattr(config,'model',object()), 'hidden_dim', 192)))
        depth = int(getattr(config, 'decoder_depth', getattr(getattr(config,'model',object()), 'decoder_depth', 3)))
        heads = int(getattr(config, 'num_heads', getattr(getattr(config,'model',object()), 'num_heads', 6)))
        drop  = float(getattr(config, 'decoder_drop_path_rate', getattr(getattr(config,'model',object()), 'decoder_drop_path_rate', 0.1)))
        self.model_type = str(getattr(config, 'diffusion_model_type', getattr(getattr(config,'diffusion',object()), 'model_type', "x_start")))
        self.output_dim = self.T * self.D
        self.dit = ErgodicDiT(self.output_dim, hd, depth, heads, dropout=drop, model_type=self.model_type)

    def forward(self, enc_out, inputs):
        enc = enc_out['encoding']    # [B, hidden_dim]
        if 'trajectories' in inputs and 'diffusion_time' in inputs:
            xt = inputs['trajectories']; t = inputs['diffusion_time']   # 训练路径：xt 和 t
            B = xt.shape[0]
            x = xt.view(B, self.output_dim)
            # 训练路径不施加起点条件
            out = self.dit(x, t, enc, conditions=None)
            return {"score": out.view(B, self.T, self.D), "diffusion_time": t}
        return {}

    @torch.no_grad()
    def inference(self, enc_out, inputs, steps=None):
        enc = enc_out['encoding']; B = enc.shape[0]
        x_T = torch.randn(B, self.output_dim, device=enc.device)
        other = {"context": enc}
        if isinstance(inputs, dict) and inputs.get('robot_state') is not None:
            other["conditions"] = {0: inputs['robot_state']}
        if steps is None:
            steps = int(getattr(getattr(self.config, "diffusion", object()), "steps", 50))
        x0 = dpm_sampler(model=self.dit, x_T=x_T, other_model_params=other, diffusion_steps=steps)
        traj = x0.view(B, self.T, self.D)
        if isinstance(inputs, dict) and inputs.get('robot_state') is not None:
            traj[:, 0, :] = inputs['robot_state']
        return {"trajectories": traj}



如何彻底杜绝实例化仓库类（不使用别名）

思路：阻断和清空这几个仓库模块的导入，并在实例化前断言当前命名 ErgodicDiffusionModel 来自 Notebook（即 module == 'main'）。这既能清除已有绑定，又能防止后续误导入把名字重绑。
在你的 Notebook 中，添加并运行一个“导入隔离”单元（放在构建模型之前，建议贴近你定义完类的 Cell 06 后面）：

导入隔离与断言（一次性执行，之后实例化）

作用：从 sys.modules 清理仓库模型模块；拦截今后对这些模块的导入；并断言当前的 ErgodicDiffusionModel 来自 Notebook。
这不会引入别名，也不会影响你允许保留的 dpm_sampler 导入。


In [ ]:
# Import quarantine: block repo model modules; ensure we use the Notebook-defined class
import sys, builtins

# 需要屏蔽的仓库模块前缀（模型/编码器/解码器/主入口）
_BLOCK = (
    "diffusion_ergodic.models.diffusion_ergodic",
    "diffusion_ergodic.models.module.encoder",
    "diffusion_ergodic.models.module.decoder",
    "diffusion_ergodic.main",
)

# 1) 清理已加载的对应模块
for m in list(sys.modules.keys()):
    if any(m == b or m.startswith(b + ".") for b in _BLOCK):
        sys.modules.pop(m, None)

# 2) 打补丁：阻止后续再次导入这些仓库模块
if not hasattr(builtins, "_orig_import_guarded"):
    builtins._orig_import_guarded = builtins.__import__
    def _guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
        if any(name == b or name.startswith(b + ".") for b in _BLOCK):
            raise ImportError(f"Blocked import of repo model module: {name}. Use the Notebook-defined class instead.")
        return builtins._orig_import_guarded(name, globals, locals, fromlist, level)
    builtins.__import__ = _guarded_import

# 3) 断言：当前命名绑定的是 Notebook 中定义的类
try:
    src_mod = ErgodicDiffusionModel.__module__
    assert src_mod == "__main__", f"ErgodicDiffusionModel is from {src_mod}, not Notebook. Re-run your class Cell 06 now."
    print("Guard OK: using Notebook-defined ErgodicDiffusionModel.")
except NameError:
    print("Define ErgodicDiffusionModel in this Notebook (Cell 06) before running the guard.")


## Cell 06 — 顶层模型（训练路径去噪；推理起点约束；默认损失）

In [255]:
# 06 ErgodicDiffusionModel (denoising training: x_t -> x_0) + masked time-weighted MSE

import torch
import torch.nn as nn

# time-weighted masked MSE（仅在有效时刻计算，末段加权）
def masked_time_weighted_mse(pred, target, gamma=4.0, eps=1e-8):
    """
    pred/target: [B, T, D]
    gamma: 时间权重幂（越大越强调末段）
    """
    B, T, D = target.shape
    mask = (target.abs().sum(dim=-1) > eps).float()          # [B,T]
    t = torch.arange(T, device=target.device).float().view(1, T)
    w = (t / max(T - 1, 1))**gamma                           # [1,T]
    w = (w * mask)                                           # [B,T]
    w = w / (w.sum(dim=1, keepdim=True) + eps)               # 仅在有效时刻归一化
    diff2 = ((pred - target)**2).sum(dim=-1)                 # [B,T]
    return ((diff2 * w).sum(dim=1).mean()) / D

class ErgodicDiffusionModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config  # 供 inference 读取采样步数等

        # SDE
        beta_min = float(getattr(getattr(config, 'diffusion', object()), 'beta_min', 0.1))
        beta_max = float(getattr(getattr(config, 'diffusion', object()), 'beta_max', 20.0))
        self.sde = VPSDE_linear(beta_min=beta_min, beta_max=beta_max)

        # 子模块
        self.encoder = ErgodicEncoder(config)
        self.decoder = ErgodicDecoder(config)

        # 仅使用 x_start 形式训练
        self.model_type = "x_start"
        self.last_loss_components = {}

    def _denorm_robot_state(self, rs: torch.Tensor) -> torch.Tensor:
        if rs is None:
            return None
        # 取出 mean/std
        mean = getattr(self.config.normalizer.robot_state, "mean", 0.0)
        std  = getattr(self.config.normalizer.robot_state, "std", 1.0)
        # 统一为 float32 并搬到 rs.device
        if isinstance(mean, torch.Tensor):
            mean = mean.to(device=rs.device, dtype=torch.float32)
        else:
            mean = torch.as_tensor(mean, dtype=torch.float32, device=rs.device)
        if isinstance(std, torch.Tensor):
            std = std.to(device=rs.device, dtype=torch.float32)
        else:
            std = torch.as_tensor(std, dtype=torch.float32, device=rs.device)
        # 调整形状并反标准化
        mean = mean.view(1, -1); std = std.view(1, -1)
        return rs * std + mean


    def forward(self, inputs, training=None):
        """
        inputs:
          distribution: [B,1,H,W]  标准化输入
          robot_state:  [B,4]      标准化初始状态
          trajectories: [B,T,4]    目标 x0（物理坐标）
          diffusion_time: [B]      U(0,1)
        训练：对 x0 加噪得到 x_t，不对 x_t 施加起点硬约束；解码器预测 x0
        """
        # 编码（Encoder 使用标准化 rs）
        enc_in = {'distribution': inputs['distribution']}
        if 'robot_state' in inputs:
            enc_in['robot_state'] = inputs['robot_state']
        enc_out = self.encoder(enc_in)

        decoder_inputs = {}
        x0 = inputs.get('trajectories')
        t  = inputs.get('diffusion_time')
        if x0 is not None and t is not None:
            B, T, D = x0.shape
            x0_flat = x0.view(B, -1)
            mean, std = self.sde.marginal_prob(x0_flat, t)
            z = torch.randn_like(mean)
            xt = (mean + std * z).view(B, T, D)
            decoder_inputs['trajectories']   = xt
            decoder_inputs['diffusion_time'] = t  # 训练路径：不传 conditions

        out = self.decoder(enc_out, decoder_inputs)
        return {
            'prediction': out.get('score', out.get('prediction')),  # 预测的 x0
            'diffusion_time': t,
            'robot_state': inputs.get('robot_state')
        }

    @torch.no_grad()
    def inference(self, inputs):
        """
        推理：Encoder 仍使用标准化 rs；
             Decoder 仅在采样时对第 0 时刻施加起点硬约束（物理坐标）
        """
        enc_out = self.encoder(inputs)
        dec_in = {}
        if inputs.get('robot_state') is not None:
            dec_in['robot_state'] = self._denorm_robot_state(inputs['robot_state'])
        steps = int(getattr(getattr(self.config, "diffusion", object()), "steps", 60))
        out = self.decoder.inference(enc_out, dec_in, steps=steps)
        return {'prediction': out.get('trajectories', out.get('prediction'))}

    def compute_loss(self, outputs, target_trajectories):
        # 默认使用时间加权 masked MSE（末段加权，忽略 padding）
        gamma = float(getattr(getattr(self.config, "training", object()), "tw_gamma", 4.0))
        main = masked_time_weighted_mse(outputs['prediction'], target_trajectories, gamma=gamma)
        self.last_loss_components = {'mse_time_weighted': float(main), 'total_loss': float(main)}
        return main

    def get_loss_components(self):
        return dict(self.last_loss_components)


## Cell 07 — 训练与评估工具（最小版）
优化器/梯度裁剪；对齐“最后有效步”的评估指标；简洁打印。

In [ ]:
# 07 Rebuild DataLoaders (single-process) + Train/Eval utilities (minimal)

import os, time, math
import numpy as np
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler

# 0) 确保使用 Notebook 中定义的 Dataset/Model（不做源码检查，避免环境限制）
assert 'ErgodicDataset' in globals(), "ErgodicDataset 未在 Notebook 中定义（请先运行数据集单元）"
assert 'ErgodicDiffusionModel' in globals(), "ErgodicDiffusionModel 未在 Notebook 中定义（请先运行模型定义单元）"

# 1) 重建 DataLoader（以单进程 num_workers=0 杀掉旧 worker；确认无误后可调大）
def build_loaders(data_dir, batch_size=64, max_trajectory_len=101, val_split=0.1,
                  num_workers=0, seed=42, transform=None):
    ds = ErgodicDataset(data_dir=data_dir, transform=transform, max_trajectory_len=max_trajectory_len)
    N  = len(ds); idx = list(range(N))
    split = int(np.floor(val_split * N))
    rng = np.random.RandomState(seed); rng.shuffle(idx)
    val_idx, train_idx = idx[:split], idx[split:]

    train_loader = DataLoader(ds, batch_size=batch_size,
                              sampler=SubsetRandomSampler(train_idx),
                              num_workers=num_workers, pin_memory=True, drop_last=False)
    val_loader   = DataLoader(ds, batch_size=batch_size,
                              sampler=SubsetRandomSampler(val_idx),
                              num_workers=num_workers, pin_memory=True, drop_last=False)
    return train_loader, val_loader

# 你可在前面 Cell 定义这些变量；没有则使用稳健默认或来自 config
def _get_cfg(path, default):
    return globals().get(path, default)

DATA_DIR   = _get_cfg("DATA_DIR",
                getattr(getattr(globals().get("config", object()), "data", object()), "data_dir",
                        "diffusion_ergodic/data/ergodic_dataset"))
BATCH_SIZE = int(_get_cfg("BATCH_SIZE",
                getattr(getattr(globals().get("config", object()), "training", object()), "batch_size", 64)))
TRAJ_LEN   = int(_get_cfg("TRAJ_LEN",
                getattr(getattr(globals().get("config", object()), "data", object()), "trajectory_len", 101)))
VAL_SPLIT  = float(_get_cfg("VAL_SPLIT",
                getattr(getattr(globals().get("config", object()), "data", object()), "validation_split", 0.1)))
NUM_WORKERS= int(_get_cfg("NUM_WORKERS", 0))  # 先 0，确认无误后可提到 2/4
SEED       = int(_get_cfg("SEED",
                getattr(getattr(globals().get("config", object()), "data", object()), "seed", 42)))
TRANSFORM  = _get_cfg("TRANSFORM", None)

train_loader, val_loader = build_loaders(
    data_dir=DATA_DIR, batch_size=BATCH_SIZE, max_trajectory_len=TRAJ_LEN,
    val_split=VAL_SPLIT, num_workers=NUM_WORKERS, seed=SEED, transform=TRANSFORM
)

# 2) 轻量自检（样本类型/形状）
s0 = train_loader.dataset[0]
for k in ("distribution","robot_state","trajectories","controls"):
    v = s0[k]
    assert isinstance(v, torch.Tensor), f"{k} 不是张量，请检查数据集单元中的 as_tensor 逻辑"
print("Sample[0]:",
      "distribution", tuple(s0["distribution"].shape),
      "robot_state",   tuple(s0["robot_state"].shape),
      "trajectories",  tuple(s0["trajectories"].shape),
      "controls",      tuple(s0["controls"].shape))

print(f"DataLoaders ready: N_train ~{len(train_loader.sampler)}, "
      f"N_val ~{len(val_loader.sampler)}, num_workers={NUM_WORKERS}")

# 3) 训练/评估工具（最小）
def build_optimizer(model, lr=1e-3, weight_decay=0.0):
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def _last_valid_idx_xy_np(xy, eps=1e-12):
    valid = ~(np.all(np.abs(xy) <= eps, axis=1))
    return np.nonzero(valid)[0][-1] if valid.any() else (xy.shape[0]-1)

@torch.no_grad()
def eval_aligned_metrics(model, loader, device, max_samples=1024):
    model.eval()
    pred_pts, gt_pts, ratios = [], [], []
    cnt = 0
    for b in loader:
        dist = b["distribution"].to(device)
        rs   = b["robot_state"].to(device)
        gt   = b["trajectories"]              # [B,T,4]（含 padding）
        out  = model.inference({"distribution": dist, "robot_state": rs})
        pred = out["prediction"].detach().cpu().numpy()
        gt_n = gt.detach().cpu().numpy()
        B = pred.shape[0]
        for i in range(B):
            k = _last_valid_idx_xy_np(gt_n[i, :, :2])
            pred_pts.append(pred[i, k, :2]); gt_pts.append(gt_n[i, k, :2])
            if k >= 2:
                steps = np.linalg.norm(pred[i, 1:k+1, :2] - pred[i, 0:k, :2], axis=-1)
                if len(steps) >= 2:
                    ratios.append(steps[-1]/(np.median(steps[:-1])+1e-8))
            cnt += 1
            if cnt >= max_samples: break
        if cnt >= max_samples: break

    pred_pts = np.asarray(pred_pts); gt_pts = np.asarray(gt_pts)
    delta = pred_pts - gt_pts
    mu = delta.mean(0)
    rmse = float(np.sqrt((delta**2).sum(1)).mean())
    std_pred = pred_pts.std(0, ddof=1); std_gt = gt_pts.std(0, ddof=1)
    tail = np.percentile(ratios, [50, 90, 99]) if len(ratios) else [np.nan]*3
    return dict(mu=mu, rmse=rmse, std_pred=std_pred, std_gt=std_gt, tail=tail)

def print_metrics(tag, m):
    print(f"[{tag}] mu={m['mu']}, rmse={m['rmse']:.4f}, "
          f"std(pred)={m['std_pred']}, std(gt)={m['std_gt']}, "
          f"tail(p50,p90,p99)={m['tail']}")


In [ ]:
# 07 Train/Eval utilities (minimal)

import os, math, time, torch, numpy as np

def build_optimizer(model, lr=1e-3, weight_decay=0.0):
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def last_valid_idx_xy_np(xy, eps=1e-12):
    # xy: [T,2] (numpy)
    valid = ~(np.all(np.abs(xy) <= eps, axis=1))
    if not valid.any(): return xy.shape[0]-1
    return np.nonzero(valid)[0][-1]

@torch.no_grad()
def eval_aligned_metrics(model, loader, device, max_samples=1024):
    model.eval()
    pred_pts, gt_pts, ratios = [], [], []
    count = 0
    for batch in loader:
        dist = batch["distribution"].to(device)
        rs   = batch["robot_state"].to(device)
        gt   = batch["trajectories"]              # [B,T,4], 物理坐标（含 padding）
        out  = model.inference({"distribution": dist, "robot_state": rs})
        pred = out["prediction"].detach().cpu().numpy()
        gt_n = gt.detach().cpu().numpy()
        B = pred.shape[0]
        for i in range(B):
            k = last_valid_idx_xy_np(gt_n[i, :, :2])
            pred_pts.append(pred[i, k, :2])
            gt_pts.append(gt_n[i, k, :2])
            if k >= 2:
                steps = np.linalg.norm(pred[i, 1:k+1, :2] - pred[i, 0:k, :2], axis=-1)
                if len(steps) >= 2:
                    ratios.append(steps[-1]/(np.median(steps[:-1])+1e-8))
            count += 1
            if count >= max_samples:
                break
        if count >= max_samples:
            break

    pred_pts = np.asarray(pred_pts); gt_pts = np.asarray(gt_pts)
    delta = pred_pts - gt_pts
    mu = delta.mean(0)
    rmse = float(np.sqrt((delta**2).sum(1)).mean())
    std_pred = pred_pts.std(0, ddof=1); std_gt = gt_pts.std(0, ddof=1)
    tail = np.percentile(ratios, [50, 90, 99]) if len(ratios) else [np.nan]*3
    return dict(mu=mu, rmse=rmse, std_pred=std_pred, std_gt=std_gt, tail=tail)

def print_metrics(tag, m):
    print(f"[{tag}] mu={m['mu']}, rmse={m['rmse']:.4f}, "
          f"std(pred)={m['std_pred']}, std(gt)={m['std_gt']}, "
          f"tail(p50,p90,p99)={m['tail']}")


## Cell 08 — 短预热训练（仅时间加权 masked MSE），周期评估
确保 compute_loss 已在 Cell 06 按“时间加权 masked MSE”实现（或在你之前的修复中已绑定）。

In [256]:
# 08 Build model + short warm-up (Notebook class only)

import gc, torch

# 可选清理旧实例，避免类定义更新后仍引用旧对象
for name in ("model", "opt"):
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 1) Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# 2) Sanity: 确保使用 Notebook 定义的类（不要 import 仓库）
assert ErgodicDiffusionModel.__module__ == "__main__", "ErgodicDiffusionModel 不是 Notebook 定义，请先重跑模型定义的 Cell 06。"

# 3) 实例化模型
model = ErgodicDiffusionModel(config).to(device)
assert hasattr(model, "config"), "model.config 未设置，请重跑 Cell 06"
assert hasattr(model.decoder.dit, "model_type"), "DIT 缺少 model_type，请检查 Cell 05 的 __init__ 传参"

model.train()

cfg_rs = model.config.normalizer.robot_state
cfg_rs.mean = (cfg_rs.mean.to(device) if isinstance(cfg_rs.mean, torch.Tensor)
               else torch.as_tensor(cfg_rs.mean, dtype=torch.float32, device=device))
cfg_rs.std  = (cfg_rs.std.to(device)  if isinstance(cfg_rs.std,  torch.Tensor)
               else torch.as_tensor(cfg_rs.std,  dtype=torch.float32, device=device))

# 4) DIT 采样所需字段检查（dpm_sampler 读取 model_type）
if not hasattr(model.decoder.dit, "model_type"):
    model.decoder.dit.model_type = "x_start"  # 或 getattr(config.diffusion, "model_type", "x_start")
assert model.decoder.dit.model_type in ("x_start","score","noise","v")

# 5) 优化器
opt = build_optimizer(model, lr=1e-3, weight_decay=0.0)

# 6) 预热：拿一个 batch 做一次前向 + 反向，验证链路
batch = next(iter(train_loader))
X = {
    "distribution": batch["distribution"].to(device),
    "robot_state":  batch["robot_state"].to(device),
    "trajectories": batch["trajectories"].to(device),
    "diffusion_time": torch.rand(batch["trajectories"].shape[0], device=device),
}
opt.zero_grad(set_to_none=True)
out  = model(X, training=True)
loss = model.compute_loss(out, X["trajectories"])
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
opt.step()

print("Prediction shape:", tuple(out["prediction"].shape), "Loss:", float(loss))

# 7) 短循环（可选 100 步）
total_steps = 100
log_every   = 25
running     = 0.0
it = iter(train_loader)

for s in range(1, total_steps+1):
    try:
        b = next(it)
    except StopIteration:
        it = iter(train_loader); b = next(it)
    X = {
        "distribution": b["distribution"].to(device),
        "robot_state":  b["robot_state"].to(device),
        "trajectories": b["trajectories"].to(device),
        "diffusion_time": torch.rand(b["trajectories"].shape[0], device=device),
    }
    opt.zero_grad(set_to_none=True)
    out  = model(X, training=True)
    loss = model.compute_loss(out, X["trajectories"])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    running += float(loss)
    if s % log_every == 0:
        comps = model.get_loss_components()
        print(f"step {s}/{total_steps}  loss={running/log_every:.4f}  comps={{k: round(v,6) for k,v in comps.items()}}")
        running = 0.0


Device: cuda
Prediction shape: (64, 101, 4) Loss: 3.5841870307922363
step 25/100  loss=0.1539  comps={k: round(v,6) for k,v in comps.items()}
step 50/100  loss=0.0409  comps={k: round(v,6) for k,v in comps.items()}
step 75/100  loss=0.0380  comps={k: round(v,6) for k,v in comps.items()}
step 100/100  loss=0.0368  comps={k: round(v,6) for k,v in comps.items()}


## Cell 09 — 正式训练（最小版：按步数早停+保存最佳）
以对齐 RMSE 作为早停指标；保存最佳 checkpoint 到 trained/best_model.pth，同时保存配置到 trained/model_config.yaml（若你在前 1–2 个 cell 里有 config 对象的话）。

In [257]:
# 09 Formal training (epoch-based) + best checkpoint by aligned RMSE

import os, yaml, torch, numpy as np
from types import SimpleNamespace

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 防护1：确保采样步数存在
if not hasattr(config, "diffusion"):
    config.diffusion = SimpleNamespace(steps=60)
else:
    if not hasattr(config.diffusion, "steps"):
        config.diffusion.steps = 60
config.diffusion.steps = int(config.diffusion.steps)

# 防护2：确保采样模型暴露 model_type
if not hasattr(model.decoder.dit, "model_type"):
    model.decoder.dit.model_type = "x_start"

# 以 epoch 语义配置
steps_per_epoch = len(train_loader)
num_epochs = int(getattr(getattr(config, "training", object()), "num_epochs", 50))
log_every_batches = max(1, steps_per_epoch // 10)  # 每 ~10 个 batch 打印一次
eval_every_epochs = 1                               # 每个 epoch 评估一次
ckpt_interval_epochs = int(getattr(getattr(config, "training", object()), "checkpoint_interval", 5))

# 优化器（可与 Cell 08 相同或重用）
opt = build_optimizer(model, lr=getattr(getattr(config, "training", object()), "learning_rate", 1e-3),
                      weight_decay=getattr(getattr(config, "training", object()), "weight_decay", 0.0))

# 保存目录
save_dir = "trained"
os.makedirs(save_dir, exist_ok=True)
best_path = os.path.join(save_dir, "best_model.pth")
cfg_path  = os.path.join(save_dir, "model_config.yaml")

# 保存一份配置（如果有）
try:
    def _to_dict(obj):
        if isinstance(obj, torch.Tensor): return obj.tolist()
        if hasattr(obj, "__dict__"): return {k: _to_dict(v) for k,v in obj.__dict__.items()}
        if isinstance(obj, (list, tuple)): return [_to_dict(v) for v in obj]
        if isinstance(obj, dict): return {k: _to_dict(v) for k,v in obj.items()}
        return obj
    with open(cfg_path, "w") as f:
        yaml.safe_dump(_to_dict(config), f)
    print(f"Saved config -> {cfg_path}")
except Exception as e:
    print(f"Skip saving config: {e}")

best_rmse = float("inf")

for epoch in range(1, num_epochs + 1):
    model.train()
    running = 0.0

    for b_idx, batch in enumerate(train_loader, start=1):
        X = {
            "distribution": batch["distribution"].to(device),
            "robot_state":  batch["robot_state"].to(device),
            "trajectories": batch["trajectories"].to(device),
            "diffusion_time": torch.rand(batch["trajectories"].shape[0], device=device),
        }
        opt.zero_grad(set_to_none=True)
        out  = model(X, training=True)
        loss = model.compute_loss(out, X["trajectories"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        running += float(loss)
        if b_idx % log_every_batches == 0 or b_idx == steps_per_epoch:
            comps = model.get_loss_components()
            comps_str = ", ".join([f"{k}={round(v,6)}" for k,v in comps.items()])
            avg_loss = running / min(log_every_batches, b_idx % log_every_batches or log_every_batches)
            print(f"epoch {epoch}/{num_epochs}  step {b_idx}/{steps_per_epoch}  "
                  f"loss={avg_loss:.4f}  comps={{ {comps_str} }}")
            running = 0.0

    # 每个 epoch 评估
    if epoch % eval_every_epochs == 0:
        m = eval_aligned_metrics(model, val_loader, device, max_samples=2048)
        print(f"[eval @ epoch {epoch}] mu={m['mu']}, rmse={m['rmse']:.4f}, "
              f"std(pred)={m['std_pred']}, std(gt)={m['std_gt']}, tail={m['tail']}")

        # 保存最优
        if m["rmse"] < best_rmse:
            best_rmse = m["rmse"]
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": opt.state_dict(),
                "metrics": m,
                "config_steps_per_epoch": steps_per_epoch
            }, best_path)
            print(f"Saved best checkpoint (rmse={best_rmse:.4f}) -> {best_path}")

    # 按 epoch 间隔保存 checkpoint
    if ckpt_interval_epochs > 0 and epoch % ckpt_interval_epochs == 0:
        ckpt_path = os.path.join(save_dir, f"checkpoint_epoch_{epoch}.pth")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": opt.state_dict(),
            "config_steps_per_epoch": steps_per_epoch
        }, ckpt_path)
        print(f"Saved checkpoint -> {ckpt_path}")


Saved config -> trained/model_config.yaml
epoch 1/200  step 140/1407  loss=0.0344  comps={ mse_time_weighted=0.036864, total_loss=0.036864 }
epoch 1/200  step 280/1407  loss=0.0309  comps={ mse_time_weighted=0.028275, total_loss=0.028275 }
epoch 1/200  step 420/1407  loss=0.0292  comps={ mse_time_weighted=0.024213, total_loss=0.024213 }
epoch 1/200  step 560/1407  loss=0.0273  comps={ mse_time_weighted=0.026187, total_loss=0.026187 }
epoch 1/200  step 700/1407  loss=0.0275  comps={ mse_time_weighted=0.024849, total_loss=0.024849 }
epoch 1/200  step 840/1407  loss=0.0264  comps={ mse_time_weighted=0.021047, total_loss=0.021047 }
epoch 1/200  step 980/1407  loss=0.0263  comps={ mse_time_weighted=0.027094, total_loss=0.027094 }
epoch 1/200  step 1120/1407  loss=0.0262  comps={ mse_time_weighted=0.029198, total_loss=0.029198 }
epoch 1/200  step 1260/1407  loss=0.0246  comps={ mse_time_weighted=0.021783, total_loss=0.021783 }
epoch 1/200  step 1400/1407  loss=0.0250  comps={ mse_time_weight

## Cell 10 — 载入最佳权重 + Sanity
- 载入 trained/best_model.pth，使 normalizer.mean/std 与模型同设备
- 设评估采样步数 steps=100，并做一次小批量推理检查

In [258]:
# 10 Load best checkpoint + set sampler steps + quick sanity

import os, torch
from types import SimpleNamespace

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

best_path = "trained/best_model.pth"
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best checkpoint from {best_path}, epoch={ckpt.get('epoch','?')}")

# ensure normalizer on same device (float32, 1D)
rsn = model.config.normalizer.robot_state
rsn.mean = torch.as_tensor(rsn.mean, dtype=torch.float32, device=device).view(-1)
rsn.std  = torch.as_tensor(rsn.std,  dtype=torch.float32, device=device).view(-1)

# set inference sampler steps
if not hasattr(model.config, "diffusion"):
    model.config.diffusion = SimpleNamespace(steps=100)
else:
    model.config.diffusion.steps = 100
print("inference steps:", model.config.diffusion.steps)

# sanity inference on a small batch
b = next(iter(val_loader))
dist = b["distribution"].to(device)
rs   = b["robot_state"].to(device)  # 假设前面训练使用了标准化 rs，这里直接传
with torch.no_grad():
    out = model.inference({"distribution": dist, "robot_state": rs})
    pred = out["prediction"]
print("Sanity pred shape:", tuple(pred.shape))
model.train(False)


/tmp/ipykernel_160577/3211505041.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(best_path, map_location=device)


Loaded best checkpoint from trained/best_model.pth, epoch=181
inference steps: 100
Sanity pred shape: (64, 101, 4)


ErgodicDiffusionModel(
  (encoder): ErgodicEncoder(
    (robot_encoder): Sequential(
      (0): Linear(in_features=4, out_features=192, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=192, out_features=384, bias=True)
    )
    (distribution_encoder): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): GELU(approximate='none')
      (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (3): GELU(approximate='none')
      (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (5): GELU(approximate='none')
    )
    (distribution_projector): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=4096, out_features=384, bias=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=384, out_features=384, bias=True)
    )
    (fusion): Sequential(
      (0): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (1): Line

In [ ]:
# Robust patch for repo ErgodicDiT: supports 'dit_blocks'/'blocks' and 'output_layer'/'final'
from types import MethodType
import torch

def dit_forward_patched(self, x, t, context, conditions=None):
    # optional start condition at t=0
    if isinstance(conditions, dict) and 0 in conditions and conditions[0] is not None:
        B = x.shape[0]
        D = conditions[0].shape[-1]
        T = self.output_dim // D
        xv = x.view(B, T, D)
        xv[:, 0, :] = conditions[0].to(xv.device)
        x = xv.view(B, self.output_dim)

    # original forward body
    h = self.traj_proj(x)
    t_emb = self.t_embedder(t)
    h = h + t_emb

    # blocks: support both names
    blocks = getattr(self, "dit_blocks", None)
    if blocks is None:
        blocks = getattr(self, "blocks", None)
    if blocks is None:
        raise AttributeError("ErgodicDiT has neither 'dit_blocks' nor 'blocks'")

    for blk in blocks:
        h = blk(h, context)

    # output layer: support both names
    out_layer = getattr(self, "output_layer", None)
    if out_layer is None:
        out_layer = getattr(self, "final", None)
    if out_layer is None:
        raise AttributeError("ErgodicDiT has neither 'output_layer' nor 'final'")

    return out_layer(h, t_emb)

# bind to the current model instance (repo class)
model.decoder.dit.forward = MethodType(dit_forward_patched, model.decoder.dit)
print("Re-patched DiT.forward (compat with repo attribute names).")


## Cell 11 — 定量评估 + 保存 metrics.yaml
- 以“最后有效步”对齐评估整集（最多 4096 样本），保存到 trained/metrics.yaml

In [259]:
# 11 Quantitative evaluation (aligned) + save metrics.yaml

import yaml, numpy as np

model.eval()
m = eval_aligned_metrics(model, val_loader, device, max_samples=4096)
print_metrics("final eval", m)

metrics = {
    "mu": m["mu"].tolist() if isinstance(m["mu"], np.ndarray) else list(m["mu"]),
    "rmse": float(m["rmse"]),
    "std_pred": m["std_pred"].tolist(),
    "std_gt": m["std_gt"].tolist(),
    "tail": m["tail"].tolist(),
    "samples": int(4096),
    "inference_steps": int(model.config.diffusion.steps),
}
os.makedirs("trained", exist_ok=True)
with open("trained/metrics.yaml", "w") as f:
    yaml.safe_dump(metrics, f)
print("Saved metrics -> trained/metrics.yaml")
model.train(False)


[final eval] mu=[-0.00026889  0.00082126], rmse=0.0055, std(pred)=[0.00661548 0.00807066], std(gt)=[0.00592329 0.00700204], tail(p50,p90,p99)=[0.18796133 0.27588996 0.35878506]
Saved metrics -> trained/metrics.yaml


ErgodicDiffusionModel(
  (encoder): ErgodicEncoder(
    (robot_encoder): Sequential(
      (0): Linear(in_features=4, out_features=192, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=192, out_features=384, bias=True)
    )
    (distribution_encoder): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): GELU(approximate='none')
      (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (3): GELU(approximate='none')
      (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (5): GELU(approximate='none')
    )
    (distribution_projector): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=4096, out_features=384, bias=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=384, out_features=384, bias=True)
    )
    (fusion): Sequential(
      (0): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (1): Line

## Cell 12 — 可视化 Pred vs GT（随机 12 条 + heatmap 背景）
- 从验证子集随机采样 12 条（seed=123），在分布热力图背景上绘制 Pred/GT，对齐“最后有效步”，保存单图与 3×4 网格图

In [1]:
# 12 Qualitative visualization: Pred vs GT with per-Gaussian components (random 12, seed=123)

import os, torch, numpy as np, matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import PowerNorm
from matplotlib.patches import Circle

# 工作空间基准（用于默认尺度）；实际仍用“动态范围”
WORKSPACE_BOUNDS = ((0.0, 3.0), (0.0, 3.0))
X0, X1 = WORKSPACE_BOUNDS[0]
Y0, Y1 = WORKSPACE_BOUNDS[1]

def last_valid_idx_xy_np(xy, eps=1e-12):
    valid = ~(np.all(np.abs(xy) <= eps, axis=1))
    return np.nonzero(valid)[0][-1] if valid.any() else (xy.shape[0]-1)

# 从验证集随机抽样 12 条
try:
    val_indices = list(val_loader.sampler.indices)
except Exception:
    val_indices = list(range(len(val_loader.dataset)))
rng = np.random.RandomState(123)
pick = rng.choice(val_indices, size=12, replace=False).tolist()

# 收集样本
dists, rs_std, gts, comp_params = [], [], [], []  # 不再使用合成网格
rsn = model.config.normalizer.robot_state
mean = rsn.mean; std = rsn.std

for idx in pick:
    s = val_loader.dataset[idx]
    dists.append(s["distribution"].unsqueeze(0))       # 仍收集，备用（本格不用）
    gts.append(s["trajectories"].unsqueeze(0))         # [1,T,4]
    rs_phys = s["robot_state"].view(1, -1).to(device=torch.device("cpu"), dtype=torch.float32)
    rs_std.append(((rs_phys - mean.cpu().view(1,-1)) / (std.cpu().view(1,-1))).unsqueeze(0))  # [1,1,4]
    gp = s["gaussian_params"]
    comp_params.append({
        "n": int(gp["n_gaussians"]),
        "centers": gp["centers"].cpu().numpy() if isinstance(gp["centers"], torch.Tensor) else gp["centers"],
        "covs":    gp["covs"].cpu().numpy()    if isinstance(gp["covs"], torch.Tensor)    else gp["covs"],
        "weights": gp["weights"].cpu().numpy() if isinstance(gp["weights"], torch.Tensor) else gp["weights"],
    })

dist_batch   = torch.cat(dists, dim=0).to(device)            # [N,1,H,W] 仅为保持接口一致（此处不使用）
rs_std_batch = torch.cat(rs_std, dim=0).squeeze(1).to(device)  # [N,4]
gt_batch     = torch.cat(gts, dim=0)                        # [N,T,4]
N = gt_batch.shape[0]

# 推理
model.eval()
with torch.no_grad():
    out = model.inference({"distribution": dist_batch, "robot_state": rs_std_batch})
pred_batch = out["prediction"].detach().cpu().numpy()
gt_batch_np = gt_batch.numpy()

# 绘图参数
pnorm = PowerNorm(gamma=0.7)  # 0.4~0.8 更突出峰值
LEVELS = 20                   # 等高线层数
GRID_RES = 150                # 分量背景网格分辨率（越大越细致）

# 输出目录
vis_dir = "trained/vis"
os.makedirs(vis_dir, exist_ok=True)

def draw_gaussian_components(ax, params, x0, x1, y0, y1):
    """
    params: dict {n, centers(10,2), covs(10), weights(10)}
    使用 z_i = w_i * exp(-c_i*((x-cx)^2+(y-cy)^2)) 绘制每个分量，不叠加
    """
    n = int(params["n"])
    C = np.asarray(params["centers"])[:n]
    V = np.asarray(params["covs"])[:n]
    W = np.asarray(params["weights"])[:n]
    # 构造物理坐标网格
    xs = np.linspace(x0, x1, GRID_RES)
    ys = np.linspace(y0, y1, GRID_RES)
    Xg, Yg = np.meshgrid(xs, ys, indexing="xy")

    for i in range(n):
        cx, cy = C[i]
        c = float(V[i])
        w = float(W[i])
        if c <= 0 or w <= 0:
            continue
        Z = w * np.exp(-c * ((Xg - cx)**2 + (Yg - cy)**2))
        # 用 contourf + contour 叠加此分量（不累加），alpha 小一些
        ax.contourf(Xg, Yg, Z, levels=LEVELS, cmap="viridis", norm=pnorm, alpha=0.6, antialiased=True)
        ax.contour( Xg, Yg, Z, levels=LEVELS//4, colors="k", linewidths=0.3, alpha=0.25)
        # 画中心与 1σ 近似圈（r = sqrt(0.5/c)）
        r = float(np.sqrt(max(1e-8, 0.5 / c)))
        ax.scatter([cx],[cy], c="yellow", edgecolors="k", s=25, zorder=3)
        circ = Circle((cx, cy), r, edgecolor="yellow", facecolor="none", lw=0.7, alpha=0.8)
        ax.add_patch(circ)

# 单图（每个样本单独 png）
for i in range(N):
    xy_gt = gt_batch_np[i, :, :2]
    xy_pr = pred_batch[i, :, :2]
    k = last_valid_idx_xy_np(xy_gt)

    # 动态范围 + 3% padding
    x_min = min(X0, xy_gt[:k+1,0].min(), xy_pr[:k+1,0].min())
    x_max = max(X1, xy_gt[:k+1,0].max(), xy_pr[:k+1,0].max())
    y_min = min(Y0, xy_gt[:k+1,1].min(), xy_pr[:k+1,1].min())
    y_max = max(Y1, xy_gt[:k+1,1].max(), xy_pr[:k+1,1].max())
    pad = 0.03 * max(X1 - X0, Y1 - Y0)
    x0, x1 = x_min - pad, x_max + pad
    y0, y1 = y_min - pad, y_max + pad

    fig, ax = plt.subplots(figsize=(4,4))
    # 绘制“每个高斯分量”的等高填色与等高线
    draw_gaussian_components(ax, comp_params[i], x0, x1, y0, y1)

    # 叠加轨迹（只到最后有效步）
    ax.plot(xy_gt[:k+1,0], xy_gt[:k+1,1], 'g-', lw=2, label="GT")
    ax.plot(xy_pr[:k+1,0], xy_pr[:k+1,1], 'r--', lw=2, label="Pred")
    ax.scatter(xy_gt[0,0], xy_gt[0,1], c='k', s=20)

    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_aspect('equal'); ax.grid(alpha=0.25)
    ax.set_title(f"sample idx={pick[i]}")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(vis_dir, f"trajectory_{pick[i]}_components.png"), dpi=170)
    plt.close(fig)

print(f"Saved individual per-component figures -> {vis_dir}")

# 3×4 网格
fig = plt.figure(figsize=(12,9))
gs = GridSpec(3, 4, figure=fig, wspace=0.25, hspace=0.25)
for i in range(N):
    ax = fig.add_subplot(gs[i//4, i%4])
    xy_gt = gt_batch_np[i, :, :2]
    xy_pr = pred_batch[i, :, :2]
    k = last_valid_idx_xy_np(xy_gt)

    x_min = min(X0, xy_gt[:k+1,0].min(), xy_pr[:k+1,0].min())
    x_max = max(X1, xy_gt[:k+1,0].max(), xy_pr[:k+1,0].max())
    y_min = min(Y0, xy_gt[:k+1,1].min(), xy_pr[:k+1,1].min())
    y_max = max(Y1, xy_gt[:k+1,1].max(), xy_pr[:k+1,1].max())
    pad = 0.03 * max(X1 - X0, Y1 - Y0)
    x0, x1 = x_min - pad, x_max + pad
    y0, y1 = y_min - pad, y_max + pad

    draw_gaussian_components(ax, comp_params[i], x0, x1, y0, y1)
    ax.plot(xy_gt[:k+1,0], xy_gt[:k+1,1], 'g-', lw=1.5)
    ax.plot(xy_pr[:k+1,0], xy_pr[:k+1,1], 'r--', lw=1.5)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_aspect('equal'); ax.grid(alpha=0.25)
    ax.set_title(str(pick[i]), fontsize=9)

plt.suptitle("Pred vs GT (random 12, seed=123) with per-Gaussian components", fontsize=12)
plt.savefig(os.path.join(vis_dir, "grid_12_components.png"), dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved grid -> {os.path.join(vis_dir, 'grid_12_components.png')}")


NameError: name 'val_loader' is not defined

## Cell 13 - 轨迹 GIF(单样本逐步动画)
若想指定样本，修改 SAMPLE_IDX = <val_index>；否则保持 None 会随机抽取。
若动画太慢，可将 STRIDE 提大（如 2 或 3）跳帧播放；或降低 GRID_RES/LEVELS。
若希望 GIF 更清晰，可把 figsize 或 dpi（在 PillowWriter 不便设置时，用 imageio 分支拼帧前 fig2.set_dpi(120)）调高。

In [277]:
# 13 Trajectory GIF (single sample, step-by-step animation with per-Gaussian background)

import os, math, numpy as np, torch, matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import PowerNorm
from matplotlib.patches import Circle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# ------------ 配置 ------------
SAMPLE_IDX = None      # 指定验证集中某条样本索引；None 表示随机抽取一条
SEED = 123             # 随机抽样种子
STEPS = int(getattr(getattr(model.config, "diffusion", object()), "steps", 100))  # 采样步数
FPS = 15               # GIF 帧率
STRIDE = 1             # 帧间步长（>1 可加快动画）
GRID_RES = 160         # 背景网格分辨率（越大越细）
LEVELS = 22            # 等高线层数
GAMMA = 0.7            # PowerNorm gamma（<1 增强峰值对比）

# ------------ 工具函数 ------------
def last_valid_idx_xy_np(xy, eps=1e-12):
    valid = ~(np.all(np.abs(xy) <= eps, axis=1))
    return np.nonzero(valid)[0][-1] if valid.any() else (xy.shape[0]-1)

def draw_gaussian_components(ax, params, x0, x1, y0, y1, grid_res=GRID_RES, levels=LEVELS, gamma=GAMMA):
    """逐分量绘制等高填色与等高线，并标注中心与 1σ 近似圈。"""
    pnorm = PowerNorm(gamma=gamma)
    n = int(params["n"])
    C = np.asarray(params["centers"])[:n]
    V = np.asarray(params["covs"])[:n]
    W = np.asarray(params["weights"])[:n]
    xs = np.linspace(x0, x1, grid_res)
    ys = np.linspace(y0, y1, grid_res)
    Xg, Yg = np.meshgrid(xs, ys, indexing="xy")
    for i in range(n):
        cx, cy = C[i]; c = float(V[i]); w = float(W[i])
        if c <= 0 or w <= 0: continue
        Z = w * np.exp(-c * ((Xg - cx)**2 + (Yg - cy)**2))
        ax.contourf(Xg, Yg, Z, levels=levels, cmap="viridis", norm=pnorm, alpha=0.7, antialiased=True)
        ax.contour( Xg, Yg, Z, levels=max(3, levels//4), colors="k", linewidths=0.3, alpha=0.25)
        r = float(np.sqrt(max(1e-8, 0.5 / c)))  # 近似 1σ
        ax.scatter([cx], [cy], c="yellow", edgecolors="k", s=25, zorder=3)
        ax.add_patch(Circle((cx, cy), r, edgecolor="yellow", facecolor="none", lw=0.7, alpha=0.9))

# ------------ 选样本 ------------
try:
    val_indices = list(val_loader.sampler.indices)
except Exception:
    val_indices = list(range(len(val_loader.dataset)))

if SAMPLE_IDX is None:
    rng = np.random.RandomState(SEED)
    SAMPLE_IDX = int(rng.choice(val_indices, size=1)[0])

s = val_loader.dataset[SAMPLE_IDX]
print("Making GIF for sample idx:", SAMPLE_IDX)

# 取输入
dist = s["distribution"].unsqueeze(0).to(device)   # [1,1,H,W]
gt   = s["trajectories"].unsqueeze(0)              # [1,T,4] (CPU)
rsn = model.config.normalizer.robot_state
rs_phys = s["robot_state"].view(1, -1).to(dtype=torch.float32, device=torch.device("cpu"))
rs_std  = ((rs_phys - rsn.mean.cpu().view(1,-1)) / (rsn.std.cpu().view(1,-1))).to(device)  # [1,4]

# 推理
with torch.no_grad():
    out = model.inference({"distribution": dist, "robot_state": rs_std})
pred = out["prediction"].detach().cpu().numpy()[0]     # [T,4]
gt_np= gt.numpy()[0]                                   # [T,4]
k = last_valid_idx_xy_np(gt_np[:, :2])

# 高斯参数
gp = s["gaussian_params"]
params = {
    "n": int(gp["n_gaussians"]),
    "centers": gp["centers"].cpu().numpy() if isinstance(gp["centers"], torch.Tensor) else gp["centers"],
    "covs":    gp["covs"].cpu().numpy()    if isinstance(gp["covs"], torch.Tensor)    else gp["covs"],
    "weights": gp["weights"].cpu().numpy() if isinstance(gp["weights"], torch.Tensor) else gp["weights"],
}

# 动态范围（含 3% 边距）
X0_def, X1_def = 0.0, 3.0
Y0_def, Y1_def = 0.0, 3.0
x_min = min(X0_def, gt_np[:k+1,0].min(), pred[:k+1,0].min())
x_max = max(X1_def, gt_np[:k+1,0].max(), pred[:k+1,0].max())
y_min = min(Y0_def, gt_np[:k+1,1].min(), pred[:k+1,1].min())
y_max = max(Y1_def, gt_np[:k+1,1].max(), pred[:k+1,1].max())
pad = 0.03 * max(X1_def - X0_def, Y1_def - Y0_def)
x0, x1 = x_min - pad, x_max + pad
y0, y1 = y_min - pad, y_max + pad

# ------------ 准备动画 ------------
fig, ax = plt.subplots(figsize=(5,5))
# 背景一次性绘制
draw_gaussian_components(ax, params, x0, x1, y0, y1)

# 轨迹对象
(gt_line,)   = ax.plot([], [], 'g-', lw=2, label="GT")
(pred_line,) = ax.plot([], [], 'r--', lw=2, label="Pred")
pred_tip = ax.scatter([], [], c='red', s=22, zorder=4)
ax.scatter([gt_np[0,0]], [gt_np[0,1]], c='k', s=20)   # 起点
ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
ax.set_aspect('equal'); ax.grid(alpha=0.25)
ax.set_title(f"sample idx={SAMPLE_IDX}")
ax.legend(loc="best", fontsize=9)

# 帧更新
T_frames = (k + 1 + STRIDE - 1) // STRIDE  # ceil((k+1)/stride)
def update(frame_idx):
    t = min(k, frame_idx * STRIDE)
    gt_line.set_data(gt_np[:t+1,0],   gt_np[:t+1,1])
    pred_line.set_data(pred[:t+1,0],  pred[:t+1,1])
    pred_tip.set_offsets([[pred[t,0], pred[t,1]]])
    return gt_line, pred_line, pred_tip

ani = FuncAnimation(fig, update, frames=T_frames, interval=1000/FPS, blit=False)
plt.close(fig)

# ------------ 保存 GIF ------------
gif_dir = os.path.join("trained", "vis", "gifs")
os.makedirs(gif_dir, exist_ok=True)
gif_path = os.path.join(gif_dir, f"trajectory_{SAMPLE_IDX}.gif")
try:
    ani.save(gif_path, writer=PillowWriter(fps=FPS))
except Exception as e:
    # 退化方案：用 imageio 手动拼帧
    import imageio.v2 as imageio
    frames = []
    fig2, ax2 = plt.subplots(figsize=(5,5))
    draw_gaussian_components(ax2, params, x0, x1, y0, y1)
    (gt_line2,)   = ax2.plot([], [], 'g-', lw=2)
    (pred_line2,) = ax2.plot([], [], 'r--', lw=2)
    pred_tip2 = ax2.scatter([], [], c='red', s=22)
    ax2.scatter([gt_np[0,0]], [gt_np[0,1]], c='k', s=20)
    ax2.set_xlim(x0, x1); ax2.set_ylim(y0, y1); ax2.set_aspect('equal'); ax2.grid(alpha=0.25)
    ax2.set_title(f"sample idx={SAMPLE_IDX}")
    for fi in range(T_frames):
        t = min(k, fi * STRIDE)
        gt_line2.set_data(gt_np[:t+1,0],   gt_np[:t+1,1])
        pred_line2.set_data(pred[:t+1,0],  pred[:t+1,1])
        pred_tip2.set_offsets([[pred[t,0], pred[t,1]]])
        fig2.canvas.draw()
        frame = np.frombuffer(fig2.canvas.tostring_rgb(), dtype=np.uint8)
        frame = frame.reshape(fig2.canvas.get_width_height()[::-1] + (3,))
        frames.append(frame)
    plt.close(fig2)
    imageio.mimsave(gif_path, frames, fps=FPS)

print("Saved GIF ->", gif_path)


Making GIF for sample idx: 47722
Saved GIF -> trained/vis/gifs/trajectory_47722.gif


## Cell 14 - 扩散过程 GIF (从噪声收敛到轨迹)

In [282]:
# 14 Diffusion path GIF (analytic) — lock start, skip-first, ease-out, hold-final, non-loop

import os, numpy as np, torch, matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import PowerNorm
from matplotlib.patches import Circle
from models.diffusion_utils.sampling import dpm_sampler  # 允许

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# ---------- 控制量 ----------
SAMPLE_IDX     = None   # None=随机选一条验证样本
SEED           = 123
STEPS_SOLVER   = int(getattr(getattr(model.config, "diffusion", object()), "steps", 100))
N_FRAMES       = 24
FPS            = 6
HOLD_FINAL_SEC = 2.0
EASE           = "ease_out"  # 或 "none"
EASE_POWER     = 0.6         # 0.4~0.7，越小末端越密集
SKIP_FIRST_NOISY_FRAMES = 2  # 跳过最前噪声帧（0=不跳）
LOCK_START     = True        # 每帧第0时刻锁定到物理起点

GRID_RES = 160
LEVELS   = 20
GAMMA    = 0.7

def last_valid_idx_xy_np(xy, eps=1e-12):
    valid = ~(np.all(np.abs(xy) <= eps, axis=1))
    return np.nonzero(valid)[0][-1] if valid.any() else (xy.shape[0]-1)

def draw_gaussian_components(ax, params, x0, x1, y0, y1, grid_res=GRID_RES, levels=LEVELS, gamma=GAMMA):
    pnorm = PowerNorm(gamma=gamma)
    n = int(params["n"])
    C = np.asarray(params["centers"])[:n]
    V = np.asarray(params["covs"])[:n]
    W = np.asarray(params["weights"])[:n]
    xs = np.linspace(x0, x1, grid_res)
    ys = np.linspace(y0, y1, grid_res)
    Xg, Yg = np.meshgrid(xs, ys, indexing="xy")
    for i in range(n):
        cx, cy = C[i]; c = float(V[i]); w = float(W[i])
        if c <= 0 or w <= 0: continue
        Z = w * np.exp(-c * ((Xg - cx)**2 + (Yg - cy)**2))
        ax.contourf(Xg, Yg, Z, levels=levels, cmap="viridis", norm=pnorm, alpha=0.7, antialiased=True)
        ax.contour( Xg, Yg, Z, levels=max(3, levels//4), colors="k", linewidths=0.3, alpha=0.25)
        r = float(np.sqrt(max(1e-8, 0.5 / c)))
        ax.scatter([cx], [cy], c="yellow", edgecolors="k", s=25, zorder=3)
        ax.add_patch(Circle((cx, cy), r, edgecolor="yellow", facecolor="none", lw=0.7, alpha=0.9))

# 选样本
try:
    val_indices = list(val_loader.sampler.indices)
except Exception:
    val_indices = list(range(len(val_loader.dataset)))
if SAMPLE_IDX is None:
    rng = np.random.RandomState(SEED)
    SAMPLE_IDX = int(rng.choice(val_indices, size=1)[0])
s = val_loader.dataset[SAMPLE_IDX]
print("Analytic diffusion GIF for sample idx:", SAMPLE_IDX)

# 输入
dist = s["distribution"].unsqueeze(0).to(device)
gt   = s["trajectories"].unsqueeze(0)              # CPU
rsn  = model.config.normalizer.robot_state
rs_phys = s["robot_state"].view(1, -1).to(dtype=torch.float32, device=torch.device("cpu"))
rs_std  = ((rs_phys - rsn.mean.cpu().view(1,-1)) / (rsn.std.cpu().view(1,-1))).to(device)

# 高斯参数
gp = s["gaussian_params"]
params = {
    "n": int(gp["n_gaussians"]),
    "centers": gp["centers"].cpu().numpy() if isinstance(gp["centers"], torch.Tensor) else gp["centers"],
    "covs":    gp["covs"].cpu().numpy()    if isinstance(gp["covs"], torch.Tensor)    else gp["covs"],
    "weights": gp["weights"].cpu().numpy() if isinstance(gp["weights"], torch.Tensor) else gp["weights"],
}

# 编码 & 固定 x_T
with torch.no_grad():
    enc_out = model.encoder({"distribution": dist, "robot_state": rs_std})
context = enc_out["encoding"]
other   = {"context": context, "conditions": {0: rs_phys.to(device)}}
gen = torch.Generator(device=device).manual_seed(SEED)
x_T = torch.randn(1, model.decoder.output_dim, device=device, generator=gen)

# 先求 x0
with torch.no_grad():
    x0_final = dpm_sampler(model=model.decoder.dit, x_T=x_T, other_model_params=other, diffusion_steps=STEPS_SOLVER)
x0_flat = x0_final.view(1, -1)

# VPSDE 解析 α(t), σ(t)
def alpha_sigma(t):
    beta_min = float(getattr(getattr(model.config,'diffusion',object()), 'beta_min', 0.1))
    beta_max = float(getattr(getattr(model.config,'diffusion',object()), 'beta_max', 20.0))
    t = torch.as_tensor(t, dtype=torch.float32, device=device)
    mean_log_coeff = -0.25 * t**2 * (beta_max - beta_min) - 0.5 * beta_min * t
    alpha_t = torch.exp(mean_log_coeff)
    sigma_t = torch.sqrt(torch.clamp(1.0 - torch.exp(2.0 * mean_log_coeff), min=1e-8))
    return alpha_t, sigma_t

with torch.no_grad():
    a_T, s_T = alpha_sigma(1.0)
    z = (x_T - a_T * x0_flat) / (s_T + 1e-8)

ts = torch.linspace(1.0, 0.0, steps=N_FRAMES, device=device)

# 生成帧（锁定起点）
frames_traj = []
rs_start_np = rs_phys.cpu().numpy()[0]
with torch.no_grad():
    for t in ts:
        a_t, s_t = alpha_sigma(t)
        x_t = a_t * x0_flat + s_t * z
        traj = x_t.view(1, model.decoder.T, model.decoder.D).detach().cpu().numpy()[0]
        if LOCK_START:
            traj[0, :rs_start_np.shape[0]] = rs_start_np  # 锁定第0时刻为物理起点
        frames_traj.append(traj)

# 跳过最前面噪声帧
if SKIP_FIRST_NOISY_FRAMES > 0 and len(frames_traj) > SKIP_FIRST_NOISY_FRAMES:
    frames_traj = frames_traj[SKIP_FIRST_NOISY_FRAMES:]

# EASE 重采样
K = len(frames_traj)
if EASE == "ease_out" and K > 2:
    t_lin = np.linspace(0.0, 1.0, K)
    t_eased = np.power(t_lin, EASE_POWER)
    idx = np.clip((t_eased * (K-1)).round().astype(int), 0, K-1)
    idx = np.unique(idx)
    frames_traj = [frames_traj[i] for i in idx]

# 末帧停留
if HOLD_FINAL_SEC > 0:
    frames_traj.extend([frames_traj[-1]] * int(max(1, HOLD_FINAL_SEC * FPS)))

# 动态范围
gt_np = gt.numpy()[0]
k = last_valid_idx_xy_np(gt_np[:, :2])
traj_final = frames_traj[-1]
x_min = min(0.0, gt_np[:k+1,0].min(), traj_final[:,0].min())
x_max = max(3.0, gt_np[:k+1,0].max(), traj_final[:,0].max())
y_min = min(0.0, gt_np[:k+1,1].min(), traj_final[:,1].min())
y_max = max(3.0, gt_np[:k+1,1].max(), traj_final[:,1].max())
pad = 0.03 * max(3.0, 3.0)
X0, X1 = x_min - pad, x_max + pad
Y0, Y1 = y_min - pad, y_max + pad

# 画动画（不循环）
fig, ax = plt.subplots(figsize=(5,5))
draw_gaussian_components(ax, params, X0, X1, Y0, Y1)
(gt_line,)   = ax.plot(gt_np[:k+1,0], gt_np[:k+1,1], 'g-', lw=2, label="GT")
(pred_line,) = ax.plot([], [], 'r--', lw=2, label="Pred")
title = ax.text(0.02, 0.98, "", ha="left", va="top", transform=ax.transAxes, fontsize=10, color="w")
ax.scatter([gt_np[0,0]], [gt_np[0,1]], c='k', s=20)
ax.set_xlim(X0, X1); ax.set_ylim(Y0, Y1)
ax.set_aspect('equal'); ax.grid(alpha=0.25)
ax.legend(loc="best", fontsize=9)

def update(fi):
    traj = frames_traj[fi]
    pred_line.set_data(traj[:k+1,0], traj[:k+1,1])
    # title.set_text(f"frame {fi+1}/{len(frames_traj)}")
    return pred_line, title

ani = FuncAnimation(fig, update,
                    frames=len(frames_traj),
                    interval=1000/FPS,
                    blit=False,
                    repeat=False)  # 不循环（Notebook 内联）

plt.close(fig)

# 保存 GIF（不循环）
gif_dir = os.path.join("trained", "vis", "gifs")
os.makedirs(gif_dir, exist_ok=True)
gif_path = os.path.join(gif_dir, f"diffusion_analytic_{SAMPLE_IDX}.gif")
try:
    ani.save(gif_path, writer=PillowWriter(fps=FPS, metadata={'loop': 1}))
except Exception as e:
    import imageio.v2 as imageio
    frames_png = []
    for fi in range(len(frames_traj)):
        fig2, ax2 = plt.subplots(figsize=(5,5))
        draw_gaussian_components(ax2, params, X0, X1, Y0, Y1)
        ax2.plot(gt_np[:k+1,0], gt_np[:k+1,1], 'g-', lw=2)
        traj = frames_traj[fi]
        ax2.plot(traj[:k+1,0], traj[:k+1,1], 'r--', lw=2)
        ax2.scatter([gt_np[0,0]], [gt_np[0,1]], c='k', s=20)
        ax2.set_xlim(X0, X1); ax2.set_ylim(Y0, Y1); ax2.set_aspect('equal'); ax2.grid(alpha=0.25)
        fig2.canvas.draw()
        frame = np.frombuffer(fig2.canvas.tostring_rgb(), dtype=np.uint8)
        frame = frame.reshape(fig2.canvas.get_width_height()[::-1] + (3,))
        frames_png.append(frame); plt.close(fig2)
    imageio.mimsave(gif_path, frames_png, fps=FPS, loop=1)

print("Saved analytic diffusion GIF ->", gif_path)

# # 同时导出 MP4（不会循环）
# try:
#     from matplotlib.animation import FFMpegWriter
#     mp4_path = os.path.join(gif_dir, f"diffusion_analytic_{SAMPLE_IDX}.mp4")
#     ani.save(mp4_path, writer=FFMpegWriter(fps=FPS, bitrate=1800))
#     print("Saved MP4 (non-looping) ->", mp4_path)
# except Exception as e:
#     print("FFMpegWriter not available:", e)


Analytic diffusion GIF for sample idx: 47722
Saved analytic diffusion GIF -> trained/vis/gifs/diffusion_analytic_47722.gif


## Cell 15 - 对比baseline

In [1]:
# Compare vs Baseline (single case): unified GMM input -> ours & baseline -> plot

import sys, os, json, subprocess, tempfile, numpy as np, matplotlib.pyplot as plt

# 允许访问 baseline 代码/脚本
BASELINE_ROOT = "/home/songxy/code/Diffusion-Ergodic/third_party/bias_search"
sys.path.insert(0, BASELINE_ROOT)

# 配置：选择接入方式（"import" 或 "subprocess"）
BASELINE_MODE = "subprocess"     # 依赖不兼容/在 docker 时用 "subprocess"
BASELINE_CLI  = "/home/songxy/code/Time-optimal-Ergodic-Control/experiments/bias_search/baseline_cli.py"
# 如果 baseline 在 time-optimal-ergodic-search 下，改成：
# BASELINE_CLI  = "/home/songxy/code/time-optimal-ergodic-search/experiments/bias_search/baseline_cli.py"

# 统一输入（示例）
centers = np.array([[1.2, 1.0],[2.6, 2.4]], np.float32)
covs    = np.array([0.15, 0.10], np.float32)      # 标量/对角均可按你们定义
weights = np.array([0.6, 0.4],  np.float32)
start   = np.array([0.1, 0.1, 0.0, 0.0], np.float32)
H = W = 32
STEPS = 80
SEED  = 123

# 生成统一 grid（如你已有 gmm_to_grid/infer_grid，请复用；下面为复刻版）
def gmm_to_grid(centers, covs, weights, H=32, W=32, workspace=(0.0,3.0,0.0,3.0)):
    x0,x1,y0,y1 = workspace
    xs = np.linspace(x0, x1, H); ys = np.linspace(y0, y1, W)
    X, Y = np.meshgrid(xs, ys, indexing="xy")
    Z = np.zeros_like(X, dtype=np.float32)
    centers = np.asarray(centers, np.float32)
    covs    = np.asarray(covs,    np.float32)
    weights = np.asarray(weights, np.float32)
    for i in range(len(weights)):
        cx, cy = centers[i]; c = covs[i]
        if np.isscalar(c): sx, sy = c, c
        elif np.ndim(c)==1: sx, sy = c[0], c[1]
        else:
            m = np.asarray(c, np.float32).reshape(2,2)
            sx, sy = np.sqrt(m[0,0]), np.sqrt(m[1,1])
        Z += weights[i] * np.exp(-(((X-cx)**2)/(2*sx**2+1e-8) + ((Y-cy)**2)/(2*sy**2+1e-8)))
    Z /= (Z.max() + 1e-8)
    return Z.astype(np.float32)

# —— ours 推理：调用你 Notebook 中的 infer_grid（若未定义，则落地一份轻量版） ——
try:
    infer_grid  # 若已定义则跳过
except NameError:
    from diffusion_ergodic.main import load_config
    from diffusion_ergodic.models.diffusion_ergodic import ErgodicDiffusionModel
    from diffusion_ergodic.models.diffusion_utils.sampling import dpm_sampler
    import torch
    CFG   = "/home/songxy/code/Diffusion-Ergodic/trained/model_config.yaml"
    CHKPT = "/home/songxy/code/Diffusion-Ergodic/trained/best_model.pth"
    DEVICE= torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg   = load_config(CFG)
    _model = ErgodicDiffusionModel(cfg).to(DEVICE); _model.eval()
    ckpt  = torch.load(CHKPT, map_location=DEVICE)
    _model.load_state_dict(ckpt["model_state_dict"]); _model.eval()
    rsn = _model.config.normalizer.robot_state
    rsn.mean = torch.as_tensor(rsn.mean, dtype=torch.float32, device=DEVICE).view(-1)
    rsn.std  = torch.as_tensor(rsn.std,  dtype=torch.float32, device=DEVICE).view(-1)

    @torch.no_grad()
    def infer_grid(grid_hw: np.ndarray, start_state: np.ndarray, steps: int=80, seed: int|None=None)->np.ndarray:
        grid_t = torch.from_numpy(np.asarray(grid_hw, np.float32)).unsqueeze(0).unsqueeze(0).to(DEVICE)
        start  = np.asarray(start_state, np.float32).reshape(1,-1)
        start_std = ((start - rsn.mean.cpu().numpy()) / rsn.std.cpu().numpy()).astype(np.float32)
        start_t = torch.from_numpy(start_std).to(DEVICE)
        enc = _model.encoder({"distribution": grid_t, "robot_state": start_t})["encoding"]
        other = {"context": enc, "conditions": {0: torch.from_numpy(start).to(DEVICE)}}
        gen = torch.Generator(device=DEVICE)
        if seed is not None: gen.manual_seed(int(seed))
        x_T = torch.randn(1, _model.decoder.output_dim, device=DEVICE, generator=gen)
        x0  = dpm_sampler(model=_model.decoder.dit, x_T=x_T, other_model_params=other, diffusion_steps=int(steps))
        traj = x0.view(1, _model.decoder.T, _model.decoder.D).detach().cpu().numpy()[0]
        return traj

# —— baseline 推理：两种模式 ——
def run_baseline_import(centers, covs, weights, start, H=32, W=32):
    # TODO: 按你们 baseline 的 API 替换以下2行
    # 例：from planner import BiasSearchPlanner; return BiasSearchPlanner(H,W).plan_from_gmm(centers,covs,weights,start)
    raise NotImplementedError("请在 run_baseline_import 内调用你们 baseline 的plan函数，并返回 [T,4]")

def run_baseline_subprocess(centers, covs, weights, start, H=32, W=32):
    # 约定 baseline_cli.py 支持以下参数，并将结果保存到 --out npz, 内含 trajectory [T,4]
    out_dir = tempfile.mkdtemp(prefix="baseline_out_")
    out_npz = os.path.join(out_dir, "traj_base.npz")
    cmd = [
        "python", BASELINE_CLI,
        "--centers", json.dumps(np.asarray(centers, np.float32).tolist()),
        "--covs",    json.dumps(np.asarray(covs,    np.float32).tolist()),
        "--weights", json.dumps(np.asarray(weights, np.float32).tolist()),
        "--start",   json.dumps(np.asarray(start,   np.float32).tolist()),
        "--H", str(H), "--W", str(W),
        "--out", out_npz
    ]
    ret = subprocess.run(cmd, check=True, capture_output=True, text=True)
    data = np.load(out_npz)
    traj = data["trajectory"]   # 需确保 baseline_cli.py 按此键保存
    return traj

# —— 生成输入，跑 ours & baseline ——
grid = gmm_to_grid(centers, covs, weights, H=H, W=W)
traj_ours = infer_grid(grid, start, steps=STEPS, seed=SEED)

if BASELINE_MODE=="import":
    traj_base = run_baseline_import(centers, covs, weights, start, H=H, W=W)
else:
    traj_base = run_baseline_subprocess(centers, covs, weights, start, H=H, W=W)

# —— 可视化 & 简单覆盖指标 ——
plt.figure(figsize=(5,5))
plt.imshow(grid.T, cmap="viridis", origin="lower", alpha=0.45, extent=(0,3,0,3), interpolation="nearest")
plt.plot(traj_ours[:,0], traj_ours[:,1], 'r--', lw=2, label="Ours")
plt.plot(traj_base[:,0], traj_base[:,1], 'g-',  lw=2, label="Baseline")
plt.scatter([start[0]],[start[1]], c='k', s=30)
plt.gca().set_aspect('equal'); plt.grid(alpha=0.3); plt.legend()
plt.title("Ours vs Baseline (unified GMM input)")
plt.show()

def coverage_min_d(traj_xy, centers):
    return np.array([np.min(np.linalg.norm(traj_xy - mu, axis=1)) for mu in centers], dtype=np.float32)

print("coverage min-distance (ours):    ", np.round(coverage_min_d(traj_ours[:,:2], centers), 3))
print("coverage min-distance (baseline):", np.round(coverage_min_d(traj_base[:,:2], centers), 3))


ModuleNotFoundError: No module named 'diffusion_ergodic.main'